# Tool Contracts and Provenance

**Docker image**: `ml4t`

An agent is only as good as what its tools return. In the forecasting system this chapter
builds, search is the single tool, so its contract decides what evidence can reach the model,
whether that evidence could have been read on the day being forecast, and whether a claim in
the final rationale can be traced back to a document. This notebook writes that contract and
the controls that sit on top of it.

**Learning Objectives**:
- Define a search tool behind a protocol so the provider can be replaced without touching
  the agent
- Filter results at a cutoff date so an agent scored on a resolved question cannot read the
  answer, and separate the undated results the filter cannot rule on
- Restrict which publishers an agent may read, and record in the execution log how much
  evidence the policy removed
- Render one tool definition into the Anthropic and OpenAI schema formats
- Read an execution log to see what a run actually called, as distinct from what the model
  said it was doing

**Book Reference**: Chapter 24, Section 24.4 (Tool Integration: Contracts, Controls,
and Context Engineering)

**Prerequisites**: [`01_react_reasoning`](01_react_reasoning.ipynb) (LLM providers
and the ReAct pattern).

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 工具契約與資料溯源

**Docker 映像檔**：`ml4t`

代理的表現取決於工具回傳的內容。在本章建立的預測系統中，搜尋是唯一工具，因此其契約決定模型能取得哪些證據、這些證據在預測當天是否已可供閱讀，以及最終理由中的主張能否追溯至文件。本 Notebook 撰寫該契約及其上層的控制機制。

**學習目標**：
- 透過協定定義搜尋工具，讓替換供應者時無須修改代理
- 依截止日期篩選結果，避免在已結算問題上受評分的代理讀到答案，並將篩選器無法判定的無日期結果分開處理
- 限制代理可閱讀哪些發布者的內容，並在執行紀錄中記下來源政策移除了多少證據
- 將同一份工具定義轉換為 Anthropic 與 OpenAI 的綱要格式
- 閱讀執行紀錄，區分實際呼叫了什麼，以及模型聲稱自己在做什麼

**書籍參考**：第 24 章，第 24.4 節（工具整合：契約、控制與上下文工程）

**先備知識**：[`01_react_reasoning`](01_react_reasoning.ipynb)（LLM 供應者與 ReAct 模式）。
<!-- ML4T_ZH_END c0000-p000 -->


In [2]:
"""Tool Contracts and Provenance - search protocol, audit trails, and domain policy."""

import sys

from utils.paths import get_chapter_dir

sys.path.insert(0, str(get_chapter_dir(24)))

import json
from datetime import date
from urllib.parse import urlparse

import polars as pl
from agent_fixtures import get_demo_question
from agent_schemas import SearchResult
from agent_tools import (
    DEFAULT_ALLOWED_DOMAINS,
    SEARCH_TOOL,
    ToolExecutor,
    create_search_client,
    format_search_results,
)

In [3]:
SEARCH_PROVIDER = "mock"  # explicitly synthetic contract fixture; "tavily" for live retrieval
MAX_RESULTS = 5

## The SearchClient Protocol

The agent built in this chapter has one tool: web search. Every provider that can serve it,
whether Tavily, a deterministic mock, or something added later, is reached through one method
signature:

```python
class SearchClient(Protocol):
    def search(
        self, query: str, max_results: int = 5, cutoff_date: date | None = None
    ) -> list[SearchResult]: ...
```

Two things in that signature are worth more than the abstraction itself. It returns typed
`SearchResult` objects rather than a provider's raw JSON, so nothing downstream depends on a
vendor's field names. And `cutoff_date` is in the contract rather than bolted on afterwards,
which is what makes point-in-time filtering the tool's job instead of every caller's.

<!-- ML4T_ZH_BEGIN c0003-p000 -->
## SearchClient 協定

本章建立的代理只有一個工具：網頁搜尋。無論是 Tavily、可重現結果的模擬用戶端，或日後新增的供應者，都透過同一個方法簽章存取：

```python
class SearchClient(Protocol):
    def search(
        self, query: str, max_results: int = 5, cutoff_date: date | None = None
    ) -> list[SearchResult]: ...
```

這個簽章有兩項比抽象化本身更重要的設計。它回傳具型別的 `SearchResult` 物件，而非供應者的原始 JSON，因此下游程式不依賴供應商的欄位名稱。此外，`cutoff_date` 被納入契約，而非事後附加，因此時間點篩選是工具的責任，不必由每個呼叫端自行處理。
<!-- ML4T_ZH_END c0003-p000 -->


### What a result carries

A `SearchResult` records where the evidence came from as well as what it says. **Provenance**
is that origin record: the URL identifies the publisher and lets a claim be traced back to
the page it came from, and `published` is the date the filtering below depends on. `score` is
the provider's own relevance ranking, which orders results and says nothing about whether
they are true.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
### 搜尋結果包含的資訊

`SearchResult` 不只記錄證據說了什麼，也記錄其來源。**資料溯源**就是這項來源紀錄：URL 指明發布者，使主張可追溯至原始頁面；`published` 則是下方篩選機制所依據的日期。`score` 是供應者自己的相關性排名，用來排列結果，並不表示結果是否屬實。
<!-- ML4T_ZH_END c0004-p000 -->


In [4]:
example = SearchResult(
    title="Fed officials signal patience on rate cuts",
    url="https://federalreserve.gov/press-releases",
    snippet="Multiple FOMC members indicate no urgency to cut rates.",
    published="2025-03-05",
    score=0.96,
)
print(f"Title:     {example.title}")
print(f"URL:       {example.url}")
print(f"Published: {example.published}")
print(f"Score:     {example.score}")

Title:     Fed officials signal patience on rate cuts
URL:       https://federalreserve.gov/press-releases
Published: 2025-03-05
Score:     0.96


## Running a Search

`SEARCH_PROVIDER` decides where results come from, and it is named rather than inferred.
Left at `mock` it builds `MockSearchClient`, whose results are a fixed dictionary in
`agent_tools.py`, so this notebook produces the same output on every machine and costs
nothing. Name a real provider, `tavily` for this chapter, and it needs that provider's key;
a missing one raises instead of quietly falling back to the fixture, which is what keeps a
synthetic result from being read as a retrieved one.

The demonstration question asks whether NVIDIA beat its Q4 FY2025 earnings expectations, which
the company answered when it reported on 2025-02-26. The question carries a **cutoff date** of
2025-02-20: the boundary the search tool enforces, dropping anything published on or after it,
so an agent answering the question reads only what existed before that day.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
## 執行搜尋

`SEARCH_PROVIDER` 決定結果來源，而且必須明確指定，不能靠推斷。保留 `mock` 時，程式會建立 `MockSearchClient`；其結果是 `agent_tools.py` 中的固定字典，因此本 Notebook 在每台電腦上都產生相同輸出，也不產生成本。若指定本章使用的真實供應者 `tavily`，就必須提供該供應者的金鑰；缺少金鑰時會拋出錯誤，而不會悄悄改用測試資料，以免把合成結果誤認為實際擷取的結果。

示範問題詢問 NVIDIA 的 2025 財年第四季財報是否優於預期；該公司於 2025-02-26 公布財報時，答案才揭曉。問題附帶的**截止日期**為 2025-02-20：搜尋工具會強制套用這條界線，移除當日或之後發布的所有內容，讓回答問題的代理只能閱讀該日期之前已存在的資訊。
<!-- ML4T_ZH_END c0006-p000 -->


In [5]:
search = create_search_client(SEARCH_PROVIDER)
print("Mode: SYNTHETIC contract fixture" if SEARCH_PROVIDER == "mock" else "Mode: LIVE retrieval")
print(f"Provider: {type(search).__name__}\n")

question = get_demo_question()
cutoff = date.fromisoformat(question.cutoff_date)
print(f"Question: {question.question}")
print(f"Cutoff:   {cutoff}\n")

results = search.search(
    "NVIDIA Q4 earnings expectations", max_results=MAX_RESULTS, cutoff_date=cutoff
)
print(f"Results: {len(results)}")
for r in results:
    print(f"  [{r.published or '?'}] {r.title}")
    if r.snippet:
        print(f"    {r.snippet[:100]}")

Mode: SYNTHETIC contract fixture
Provider: MockSearchClient

Question: Will NVIDIA beat Q4 FY2025 earnings expectations?
Cutoff:   2025-02-20

Results: 2
  [2025-02-17] Synthetic fixture: NVIDIA suppliers signal continued AI server demand
    Supply-chain checks point to resilient GPU demand ahead of earnings report.
  [2025-02-18] Synthetic fixture: Analysts raise NVIDIA targets ahead of earnings
    Street revisions remain constructive ahead of the Feb 26 report.


## Point-in-Time Filtering

The question above asks about an event that has already resolved, which is the only kind of
question a forecasting agent can be scored on. It is also the kind that is trivially easy to
get right by accident: search the open web today for "did NVIDIA beat Q4 FY2025 earnings" and
the first result answers it. An agent evaluated that way is measuring the search index, not
forecasting.

`cutoff_date` is the guard. The client drops any result whose publication date is on or after
the cutoff, so what reaches the agent is dated strictly earlier than that day. The question's
own `cutoff_date` is 2025-02-20 and NVIDIA reported on 2025-02-26, so the agent works from
what was published up to and including 2025-02-19 and the answer arrives a week later.

The filter keeps a result only when it carries a publication date that parses and falls
before the cutoff, so an undated item is dropped rather than trusted. What it cannot do is
establish what a page said on the date it claims, or remove what the model already knows from
training. A cutoff makes the retrieval point-in-time; it does not make the forecast one.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 時間點篩選

上述問題涉及已經結算的事件，這是預測代理唯一能據以評分的問題類型。這類問題也很容易意外答對：今天在公開網路上搜尋「NVIDIA 2025 財年第四季財報是否優於預期」，第一筆結果就會給出答案。如此評估代理，衡量的是搜尋索引，而非預測能力。

`cutoff_date` 是防護機制。用戶端會移除發布日期在截止日當天或之後的結果，因此代理取得的資料日期嚴格早於截止日。問題本身的 `cutoff_date` 為 2025-02-20，而 NVIDIA 於 2025-02-26 公布財報；所以代理只能使用截至 2025-02-19（含當日）已發布的資訊，答案則在約一週後才出現。

篩選器只保留發布日期可解析且早於截止日的結果，因此沒有日期的項目會被移除，而不會獲得信任。但它無法確認網頁在聲稱的日期實際寫了什麼，也無法抹除模型從訓練資料中已知的內容。截止日期讓資訊擷取符合特定時間點，卻不能單靠它保證預測本身符合該時間點。
<!-- ML4T_ZH_END c0008-p000 -->


In [6]:
# Search WITHOUT cutoff - all results returned
all_results = search.search("NVIDIA Q4 earnings", max_results=MAX_RESULTS)
print(f"Without cutoff: {len(all_results)} results")
for r in all_results:
    print(f"  [{r.published or '?'}] {r.title}")

print()

# Search WITH cutoff - future results filtered out
filtered_results = search.search("NVIDIA Q4 earnings", max_results=MAX_RESULTS, cutoff_date=cutoff)
print(f"With cutoff ({cutoff}): {len(filtered_results)} results")
for r in filtered_results:
    print(f"  [{r.published or '?'}] {r.title}")

Without cutoff: 3 results
  [2025-02-17] Synthetic fixture: NVIDIA suppliers signal continued AI server demand
  [2025-02-18] Synthetic fixture: Analysts raise NVIDIA targets ahead of earnings
  [2025-02-20] Synthetic fixture: Investors watch valuation risk as AI rally broadens

With cutoff (2025-02-20): 2 results
  [2025-02-17] Synthetic fixture: NVIDIA suppliers signal continued AI server demand
  [2025-02-18] Synthetic fixture: Analysts raise NVIDIA targets ahead of earnings


## Formatting Results for the Agent

Results reach the model as text in a tool message, so the formatting is part of the contract
too. `format_search_results` numbers each result and puts the title, URL, snippet and
publication date on their own lines. The URL and the date are there so the model can weigh a
primary source against a blog and recent reporting against stale reporting, and so that a
quote in the final rationale can be traced back to the document it came from.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
## 格式化結果供代理使用

結果會以工具訊息中的文字傳給模型，因此格式也是契約的一部分。`format_search_results` 為每項結果編號，並將標題、URL、摘要片段及發布日期各自放在獨立一行。URL 與日期讓模型能權衡第一手來源和部落格，以及新近報導和過時報導；也讓最終理由中的引述可追溯至原始文件。
<!-- ML4T_ZH_END c0010-p000 -->


**Finding**: The synthetic filter excludes records outside its date bound.
A historical simulation also needs evidence available at its forecast cutoff
and an assessment of the model's prior knowledge. These fixture results test
the filter, not historical forecasting skill.

<!-- ML4T_ZH_BEGIN c0011-p000 -->
**發現**：合成資料篩選器會排除日期超出界線的紀錄。歷史模擬還需要確保證據在預測截止時已可取得，並評估模型先前已知的資訊。這些測試資料檢驗的是篩選器，不是歷史預測能力。
<!-- ML4T_ZH_END c0011-p000 -->


In [7]:
formatted = format_search_results(filtered_results)
print(formatted[:500])

SEARCH RESULTS:
1. Synthetic fixture: NVIDIA suppliers signal continued AI server demand
   URL: https://example.org/nvidia-0
   Supply-chain checks point to resilient GPU demand ahead of earnings report.
   Published: 2025-02-17

2. Synthetic fixture: Analysts raise NVIDIA targets ahead of earnings
   URL: https://example.org/nvidia-1
   Street revisions remain constructive ahead of the Feb 26 report.
   Published: 2025-02-18



## Execution Audit Trail

`ToolExecutor` wraps the search client and records every call: the query, whether it
succeeded, how long it took, and which client served it. That record is kept apart from the
agent's reasoning trace on purpose. The reasoning trace is what the model said it was doing;
the execution log is what happened. Debugging a bad forecast usually starts with the
difference between them.

<!-- ML4T_ZH_BEGIN c0013-p000 -->
## 執行稽核紀錄

`ToolExecutor` 包裝搜尋用戶端並記錄每次呼叫：查詢內容、是否成功、耗時，以及由哪個用戶端執行。這份紀錄刻意與代理的推理軌跡分開。推理軌跡記錄模型聲稱自己在做什麼；執行紀錄記錄實際發生的事。排查不良預測時，通常從兩者的差異著手。
<!-- ML4T_ZH_END c0013-p000 -->


In [8]:
executor = ToolExecutor(search=search)

# Execute several searches
r1 = executor.execute_search("NVIDIA earnings Q4 2025", max_results=3, cutoff_date=cutoff)
r2 = executor.execute_search("semiconductor demand AI servers", max_results=3, cutoff_date=cutoff)
r3 = executor.execute_search("NVIDIA valuation risk", max_results=3, cutoff_date=cutoff)

print(f"Searches executed: {len(executor.execution_log)}\n")

pl.DataFrame(
    {
        "query": [entry.args.get("query", "?")[:38] for entry in executor.execution_log],
        "status": [entry.status for entry in executor.execution_log],
        "duration_ms": [round(entry.duration_ms, 1) for entry in executor.execution_log],
        "result_preview": [entry.result_preview for entry in executor.execution_log],
    }
)

Searches executed: 3



query,status,duration_ms,result_preview
str,str,f64,str
"""NVIDIA earnings Q4 2025""","""success""",0.2,"""2 results"""
"""semiconductor demand AI server…","""success""",0.0,"""2 results"""
"""NVIDIA valuation risk""","""success""",0.0,"""2 results"""


The elapsed times round to zero because the mock client returns from a dictionary in memory.
Against a live provider the same column carries the network round trip, which is where an
agent spends most of its wall-clock time and where a per-call timeout has to be set.

<!-- ML4T_ZH_BEGIN c0015-p000 -->
經過時間四捨五入後為零，因為模擬用戶端直接從記憶體中的字典回傳結果。若使用即時供應者，同一欄會記錄網路來回傳輸的時間；代理大部分的實際耗時都在這裡，因此也需要為每次呼叫設定逾時上限。
<!-- ML4T_ZH_END c0015-p000 -->


## Source Policy

Which publishers an agent may read is a research decision, not a prompt detail. An allowlist
names the domains whose reporting counts as evidence; a blocklist names the ones that do not.
`apply_domain_policy` in `agent_tools.py` implements both, matching on the registered domain
and its subdomains after stripping a leading `www.`, so `reuters.com` also covers
`uk.reuters.com`. The blocklist is applied first, then the allowlist.

`DEFAULT_ALLOWED_DOMAINS` is the chapter's starting set: wire services, financial newspapers,
and the primary sources whose numbers the others report on.

<!-- ML4T_ZH_BEGIN c0016-p000 -->
## 來源政策

代理可以閱讀哪些發布者的內容，是研究決策，而非提示詞細節。允許清單列出可作為證據的網域；封鎖清單列出不可採用的網域。`apply_domain_policy` 位於 `agent_tools.py`，同時實作兩者：它會先移除開頭的 `www.`，再依註冊網域及其子網域比對，因此 `reuters.com` 也涵蓋 `uk.reuters.com`。程式先套用封鎖清單，再套用允許清單。

`DEFAULT_ALLOWED_DOMAINS` 是本章的初始集合：通訊社、財經報紙，以及其他來源所報導數字的第一手資料來源。
<!-- ML4T_ZH_END c0016-p000 -->


**Observation**: These synthetic calls demonstrate the audit record, not
provider latency. Inspect measured durations from a genuine live capture to
identify slow queries; a fixture's timing cannot predict a live service.

<!-- ML4T_ZH_BEGIN c0017-p000 -->
**觀察**：這些合成呼叫示範稽核紀錄，而非供應者延遲。應檢視真實即時紀錄測得的耗時，以找出緩慢查詢；測試資料的時間無法預測即時服務的表現。
<!-- ML4T_ZH_END c0017-p000 -->


## Domain policy

The tool executor filters returned URLs by hostname before providing them to the
agent. This is an access constraint, not evidence that a retained source is true.
The next local example is a labeled synthetic domain-filter fixture.

<!-- ML4T_ZH_BEGIN c0018-p000 -->
## 網域政策

工具執行器會依主機名稱篩選回傳的 URL，再將結果提供給代理。這是一項存取限制，不能據此認定保留下來的來源屬實。接下來的本機範例是明確標示的合成網域篩選測試資料。
<!-- ML4T_ZH_END c0018-p000 -->


In [9]:
assert len(DEFAULT_ALLOWED_DOMAINS) == 11, "DEFAULT_ALLOWED_DOMAINS count drifted"
pl.DataFrame({"allowed_domain": sorted(DEFAULT_ALLOWED_DOMAINS)})

allowed_domain
str
"""bloomberg.com"""
"""bls.gov"""
"""cmegroup.com"""
"""cnbc.com"""
"""coindesk.com"""
…
"""ft.com"""
"""nasdaq.com"""
"""reuters.com"""


### Custom domain policy - observable enforcement

Policy enforcement here is **post-retrieval**: the search provider returns
whatever matches the query, and the agent's tool layer filters results
whose registered domain is not on the allowlist (or appears on a
blocklist) before handing them back. The fixture below seeds a mock
search whose results contain a `reddit.com` URL alongside a `wsj.com`
URL; instantiate two policies and watch the blocked one disappear.

<!-- ML4T_ZH_BEGIN c0020-p000 -->
### 自訂網域政策：可觀察的強制執行

此處在**擷取之後**執行政策：搜尋供應者回傳符合查詢的所有內容，代理的工具層再依註冊網域篩選，將未列入允許清單（或列入封鎖清單）的結果移除，然後才交給代理。下方測試資料為模擬搜尋預先放入一個 `reddit.com` URL 和一個 `wsj.com` URL；建立兩套政策，觀察被封鎖的結果如何消失。
<!-- ML4T_ZH_END c0020-p000 -->


In [10]:
mixed_results = [
    SearchResult(
        title="Synthetic Fed reaction on rates",
        url="https://www.reuters.com/markets/fed-reaction",
        snippet="...",
        published="2025-03-04",
        score=0.91,
    ),
    SearchResult(
        title="Synthetic WSJ analysis: NVIDIA earnings setup",
        url="https://www.wsj.com/articles/nvidia",
        snippet="...",
        published="2025-03-05",
        score=0.88,
    ),
    SearchResult(
        title="Synthetic r/wallstreetbets: NVDA hot take",
        url="https://www.reddit.com/r/wallstreetbets/x",
        snippet="...",
        published="2025-03-05",
        score=0.40,
    ),
]


def apply_domain_policy(
    items: list[SearchResult],
    *,
    allowed: set[str] | None = None,
    blocked: set[str] | None = None,
) -> list[SearchResult]:
    """Drop results whose host is not on `allowed`, or is on `blocked`."""
    keep: list[SearchResult] = []
    for r in items:
        host = urlparse(r.url).netloc.removeprefix("www.")
        if blocked and host in blocked:
            continue
        if allowed and host not in allowed:
            continue
        keep.append(r)
    return keep


financial_only = apply_domain_policy(
    mixed_results, allowed={"reuters.com", "wsj.com", "ft.com", "bloomberg.com"}
)
no_social = apply_domain_policy(mixed_results, blocked={"reddit.com", "twitter.com"})

pl.DataFrame(
    {
        "policy": ["raw retrieval", "allowlist (financial)", "blocklist (social)"],
        "n_results": [len(mixed_results), len(financial_only), len(no_social)],
        "hosts_kept": [
            ", ".join(urlparse(r.url).netloc for r in mixed_results),
            ", ".join(urlparse(r.url).netloc for r in financial_only),
            ", ".join(urlparse(r.url).netloc for r in no_social),
        ],
    }
)

policy,n_results,hosts_kept
str,i64,str
"""raw retrieval""",3,"""www.reuters.com, www.wsj.com, …"
"""allowlist (financial)""",2,"""www.reuters.com, www.wsj.com"""
"""blocklist (social)""",2,"""www.reuters.com, www.wsj.com"""


## Tool Schema Translation

The `ToolDefinition` class renders the search tool schema to both Anthropic and
OpenAI formats. This is how the LLM knows what tools are available and how to
invoke them. Same definition, different provider formats.

<!-- ML4T_ZH_BEGIN c0022-p000 -->
## 工具綱要轉換

`ToolDefinition` 類別能將搜尋工具綱要轉換成 Anthropic 與 OpenAI 兩種格式。LLM 藉此得知有哪些工具可用，以及如何呼叫。定義相同，供應者格式不同。
<!-- ML4T_ZH_END c0022-p000 -->


In [11]:
print(f"Tool: {SEARCH_TOOL.name}")
print(f"Description: {SEARCH_TOOL.description}")
print(f"Parameters: {json.dumps(SEARCH_TOOL.parameters, indent=2)}")
print(f"Required: {SEARCH_TOOL.required}")

Tool: search
Description: Search the web for recent news, analysis, and data relevant to a forecasting question.
Parameters: {
  "query": {
    "type": "string",
    "description": "Search query string"
  }
}
Required: ['query']


### Anthropic format

Anthropic uses `input_schema` at the top level.

<!-- ML4T_ZH_BEGIN c0024-p000 -->
### Anthropic 格式

Anthropic 在最上層使用 `input_schema`。
<!-- ML4T_ZH_END c0024-p000 -->


In [12]:
print(json.dumps(SEARCH_TOOL.to_anthropic_schema(), indent=2))

{
  "name": "search",
  "description": "Search the web for recent news, analysis, and data relevant to a forecasting question.",
  "input_schema": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string",
        "description": "Search query string"
      }
    },
    "required": [
      "query"
    ]
  }
}


### OpenAI format

OpenAI wraps the schema inside a `function` object with a `type: "function"` envelope.

<!-- ML4T_ZH_BEGIN c0026-p000 -->
### OpenAI 格式

OpenAI 將綱要放在 `function` 物件之中，並以 `type: "function"` 作為外層結構。
<!-- ML4T_ZH_END c0026-p000 -->


In [13]:
print(json.dumps(SEARCH_TOOL.to_openai_schema(), indent=2))

{
  "type": "function",
  "function": {
    "name": "search",
    "description": "Search the web for recent news, analysis, and data relevant to a forecasting question.",
    "parameters": {
      "type": "object",
      "properties": {
        "query": {
          "type": "string",
          "description": "Search query string"
        }
      },
      "required": [
        "query"
      ]
    }
  }
}


**Observation**: The structural difference is minor (Anthropic uses `input_schema`,
OpenAI wraps in a `function` object). Provider-agnostic tool definitions prevent
provider-specific schema mismatches.

<!-- ML4T_ZH_BEGIN c0028-p000 -->
**觀察**：結構差異不大（Anthropic 使用 `input_schema`，OpenAI 則包在 `function` 物件中）。與供應者無關的工具定義可避免不同供應者的綱要彼此不符。
<!-- ML4T_ZH_END c0028-p000 -->


## When the Tool Is Unavailable

An agent deployed inside a network with no outbound access, or run while a provider is down,
still has to do something. `ToolExecutor` treats an absent client as a logged outcome rather
than an exception: the call is recorded with status `disabled` and returns no results, so the
agent's loop continues and the record shows why the evidence is missing. An exception here
would abort the run and leave nothing to audit.

<!-- ML4T_ZH_BEGIN c0029-p000 -->
## 工具無法使用時

部署於無法連接外部網路的環境，或供應者服務中斷時，代理仍須有所處理。`ToolExecutor` 將缺少用戶端視為要記錄的結果，而非例外：呼叫會以 `disabled` 狀態記錄，並回傳空結果，讓代理迴圈繼續執行，也讓紀錄說明證據缺失的原因。若在這裡拋出例外，執行就會中止，無從稽核。
<!-- ML4T_ZH_END c0029-p000 -->


In [14]:
disabled_executor = ToolExecutor(search=None)
empty = disabled_executor.execute_search("test query")
print(f"Results when disabled: {len(empty)}")
print(f"Log entry: {disabled_executor.execution_log[0].status}")

Results when disabled: 0
Log entry: disabled


## Key Takeaways

1. **The tool contract decides what the agent can know.** Everything downstream, the forecast
   included, is a function of what search returned, so the contract is worth as much design
   attention as the prompt.
2. **A publication-date filter is what separates a backtest from a demonstration.** Without
   it an agent evaluated on a resolved question reads the answer.
3. **A result with no date is not a result before the cutoff.** Keep the two apart in the
   record and decide the policy explicitly; silently counting undated evidence as
   point-in-time safe is how a hindsight-free claim goes wrong.
4. **Enforce source policy in the tool layer rather than the prompt.** A model asked not to read
   social media sometimes reads it anyway; a host that never reaches the model cannot.
5. **The execution log is the record of what ran**, separate from the model's account of what
   it was doing. When those two disagree, the log is the one to trust.
6. **One tool definition, rendered per provider.** The schema differences between Anthropic
   and OpenAI are mechanical, and a single `ToolDefinition` keeps them from becoming two
   sources of truth that drift.

**Known limitations of what is built here.** Source policy runs after retrieval, so a blocked
document was still fetched and paid for; a provider-side domain filter is cheaper where the
API offers one. The allowlist is a list of publishers, which is a crude proxy for evidence
quality: a syndicated wire story on an allowed domain and the same story on a blocked one are
the same evidence. Nothing here deduplicates results that repeat one underlying source, so an
agent can mistake five copies of one story for five independent confirmations.


<!-- ML4T_ZH_BEGIN c0031-p000 -->
## 重點整理

1. **工具契約決定代理能知道什麼。** 包括預測在內的所有下游結果，都取決於搜尋回傳的內容，因此契約值得投入與提示詞同等的設計心力。
2. **發布日期篩選器區分了回測與示範。** 若沒有這項篩選，代理在已結算問題上受評估時，就會讀到答案。
3. **沒有日期的結果不等於截止日前的結果。** 應在紀錄中分開標示，並明確決定政策；若默默將無日期證據視為符合時間點要求，聲稱沒有事後資訊的結論就可能出錯。
4. **在工具層而非提示詞中執行來源政策。** 即使要求模型不要閱讀社群媒體，它仍可能這麼做；若某個來源從未傳給模型，模型就不可能閱讀。
5. **執行紀錄記載了實際執行的內容**，與模型對自己行為的描述分開。兩者不一致時，應以執行紀錄為準。
6. **一份工具定義，依供應者轉換格式。** Anthropic 與 OpenAI 的綱要差異只是格式上的；單一 `ToolDefinition` 可避免產生兩套逐漸偏離的定義。

**此處實作的已知限制。** 來源政策在擷取後才執行，因此遭封鎖的文件仍已被擷取並計費；若 API 提供供應者端網域篩選，使用它會更省成本。允許清單列出的是發布者，僅能粗略代表證據品質：允許網域上的通訊社轉載報導，與封鎖網域上的同一篇報導，其實是同一份證據。此處沒有將指向同一底層來源的重複結果去重，因此代理可能把同一篇報導的五份複本誤認為五項獨立佐證。
<!-- ML4T_ZH_END c0031-p000 -->

**Next**: [`03_state_and_memory`](03_state_and_memory.ipynb) makes the agent's state explicit
so a run can be checkpointed, replayed, and gated on evidence quality.

**Book**: Section 24.4 covers tool contract design, the Model Context Protocol, and
sandboxing patterns.

<!-- ML4T_ZH_BEGIN c0031-p001 -->
**下一步**：[`03_state_and_memory`](03_state_and_memory.ipynb) 會明確定義代理狀態，讓執行過程可以建立檢查點、重播，並根據證據品質設置門檻。

**書籍**：第 24.4 節涵蓋工具契約設計、Model Context Protocol，以及沙盒化模式。
<!-- ML4T_ZH_END c0031-p001 -->
